In [1]:
import torchvision.models as models
import torch.nn as nn

def get_model(model_name, num_classes, freeze_base=True):
    """
    Factory function to initialize different architectures.
    """
    print(f"Initializing {model_name}...")
    
    if model_name == 'resnet34':
        weights = models.ResNet34_Weights.IMAGENET1K_V1
        model = models.resnet34(weights=weights)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'efficientnet_b2':
        # EfficientNet-B2 is great for fine-grained details
        weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
        model = models.efficientnet_b2(weights=weights)
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'densenet121':
        # DenseNet-121 is known for feature reuse in medical imaging
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        model = models.densenet121(weights=weights)
        num_ftrs = model.classifier.in_features
        model.classifier = nn.Linear(num_ftrs, num_classes)
    
    else:
        raise ValueError("Model name not supported.")

    # Generic freezing logic
    if freeze_base:
        for param in model.parameters():
            param.requires_grad = False
            
        # Unfreeze the head (classifier/fc)
        if hasattr(model, 'fc'):
            for param in model.fc.parameters(): param.requires_grad = True
        elif hasattr(model, 'classifier'):
            for param in model.classifier.parameters(): param.requires_grad = True
            
    return model

In [2]:
# ==========================================
# 1. Imports & Device Configuration
# ==========================================
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

# Set device to GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Number of classes in our HAM10000 dataset
NUM_CLASSES = 7

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available (GPU) : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Graphics card : {torch.cuda.get_device_name(0)}")

Using device: cuda
PyTorch version: 2.7.1+cu118
CUDA available (GPU) : True
Graphics card : NVIDIA GeForce GTX 1650


In [3]:
# ==========================================
# 1.5. RE-CREATING DATA LOADERS
# ==========================================
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

# 1. Load Paths and Dataframe
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
IMAGE_DIR = DATA_DIR / 'all_images'
METADATA_PATH = DATA_DIR / 'HAM10000_metadata.csv'

df = pd.read_csv(METADATA_PATH)
df['path'] = df['image_id'].apply(lambda x: IMAGE_DIR / f"{x}.jpg")

# 2. Re-define Dataset Class and Transforms
class SkinCancerDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.label_map = {'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label_str = self.df.iloc[idx]['dx']
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        import torch # Ensure torch is available here
        return image, torch.tensor(self.label_map[label_str], dtype=torch.long)

NORM_MEAN, NORM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), transforms.RandomRotation(20),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)
])

# 3. Stratified Split and DataLoaders creation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['dx'])

train_dataset = SkinCancerDataset(train_df, transform=train_transforms)
val_dataset = SkinCancerDataset(val_df, transform=val_transforms)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"DataLoaders successfully loaded! Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

DataLoaders successfully loaded! Train batches: 251, Val batches: 63


In [6]:
# 1. Import (sans class_weights)
from utils import train_model_advanced, FocalLoss, create_balanced_train_loader
import torch
import numpy as np

# On fabrique notre loader équilibré
train_loader_balanced = create_balanced_train_loader(train_df, train_dataset, BATCH_SIZE)

# 2. Recalcul dynamique des poids dans le notebook
class_counts = train_df['dx'].value_counts().sort_index().values
class_weights = 1.0 / class_counts

# 3. Instanciation de la Focal Loss
alpha_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion_focal = FocalLoss(alpha=alpha_tensor, gamma=2.0)

# List of models to test
models_to_benchmark = ['efficientnet_b2', 'densenet121']
benchmark_results = {}

for m_name in models_to_benchmark:
    # 1. Initialize
    model = get_model(m_name, NUM_CLASSES, freeze_base=True)
    model = model.to(device)
    
    # 2. Setup Optimizer (Only for the unfreezed head)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    
    # --- CORRECTION ICI : Recréation du Scheduler ---
    from torch.optim import lr_scheduler
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
    
    # 3. Train
    print(f"\n--- Training {m_name} ---")
    best_model, val_labels, val_preds = train_model_advanced(
        model=model,
        criterion=criterion_focal, # Utilise l'instance créée juste au-dessus
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader_balanced, 
        val_loader=val_loader,
        device=device,
        num_epochs=15, 
        early_stopping_patience=4
    )
    
    # 4. Save model with unique name
    save_path = f"../models/{m_name}_focal_benchmark.pth"
    torch.save(best_model.state_dict(), save_path)
    
    # 5. Store Metrics for comparison
    from sklearn.metrics import classification_report
    target_names = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
    report = classification_report(val_labels, val_preds, target_names=target_names, output_dict=True, zero_division=0)
    benchmark_results[m_name] = {
        'accuracy': report['accuracy'],
        'mel_recall': report['mel']['recall'],
        'mel_precision': report['mel']['precision'],
        'mel_f1': report['mel']['f1-score']
    }

# Final Comparison Table
import pandas as pd
df_results = pd.DataFrame(benchmark_results).T
print("\n=== FINAL BENCHMARK RESULTS ===")
print(df_results)

Setting up WeightedRandomSampler...
Balanced DataLoader ready.
Initializing efficientnet_b2...

--- Training efficientnet_b2 ---

Epoch 1/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0024
Val Loss: 0.0022
🌟 New best model found! Loss: 0.0022

Epoch 2/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0017
Val Loss: 0.0018
🌟 New best model found! Loss: 0.0018

Epoch 3/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0016
Val Loss: 0.0018
Early stopping counter: 1 out of 4

Epoch 4/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0016
Val Loss: 0.0017
🌟 New best model found! Loss: 0.0017

Epoch 5/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0015
Val Loss: 0.0017
🌟 New best model found! Loss: 0.0017

Epoch 6/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0015
Val Loss: 0.0018
Early stopping counter: 1 out of 4

Epoch 7/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0015
Val Loss: 0.0019
Early stopping counter: 2 out of 4

E

100.0%



--- Training densenet121 ---

Epoch 1/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0025
Val Loss: 0.0027
🌟 New best model found! Loss: 0.0027

Epoch 2/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0017
Val Loss: 0.0019
🌟 New best model found! Loss: 0.0019

Epoch 3/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0016
Val Loss: 0.0016
🌟 New best model found! Loss: 0.0016

Epoch 4/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0014
Val Loss: 0.0019
Early stopping counter: 1 out of 4

Epoch 5/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0014
Val Loss: 0.0019
Early stopping counter: 2 out of 4

Epoch 6/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0014
Val Loss: 0.0013
🌟 New best model found! Loss: 0.0013

Epoch 7/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0013
Val Loss: 0.0016
Early stopping counter: 1 out of 4

Epoch 8/15
----------
Current Learning Rate: 0.001
Train Loss: 0.0013
Val Loss: 0.0018
Early stoppin

### 